**A500 variant** — generated from the SP100 notebook by `scripts/clone_notebooks_a500.py`; do not edit directly, edit the original and re-run the script.

Requires `data/A500/raw/` built by `scripts/preprocess_a500.py` (replaces notebooks 1-2 for A500).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader

from datasets.SP100Stocks import SP100Stocks
from notebooks.models import A3TGCN, train, get_regression_error, plot_regression

# Stock prices forecasting
The goal of this task is to predict the (normalized) price at the timestep $t+1$ for each stock in the A500 index. For this task, we use the previously introduced Spatio-Temporal Graph Neural Networks.

## Loading the data
The data from the custom PyG dataset for forecasting is loaded into a PyTorch dataloader.

In [ ]:
seq_len = 25
dataset = SP100Stocks(root="../data/A500/", past_window=seq_len, force_reload=True)
dataset, dataset[0]

In [ ]:
train_part = .9
batch_size = 32

train_dataset, test_dataset = dataset[:int(train_part * len(dataset))], dataset[int(train_part * len(dataset)):]
print(f"Train dataset: {len(train_dataset)}, Test dataset: {len(test_dataset)}")
train_dataloader, test_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True), DataLoader(test_dataset, batch_size=len(test_dataset), drop_last=True)

## Training
The previously implemented models are used, trained using the training dataset and the Adam optimizer. The `weight_decay` parameter is used for L2 regularization, to follow the T-GCN papers methodology. The loss is calculated using the Mean Squared Error (MSE) loss function.

In [ ]:
in_channels, out_channels, hidden_size, layers_nb = dataset[0].x.shape[-2], 1, 16, 2
model = A3TGCN(in_channels, out_channels, hidden_size, layers_nb, use_gat=False)

lr, weight_decay, num_epochs = 0.005, 1e-5, 16

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
model

In [ ]:
train(model, optimizer, criterion, train_dataloader, test_dataloader, num_epochs, "PriceForecasting")

## Results
The model tries to forecast the variations one timestep ahead for four stocks. The real values are plotted against the forecasted values.

### Results on train data

In [ ]:
mse, rmse, mae, mre = get_regression_error(model, train_dataloader)
print(f"Train MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, MRE: {mre:.4f}")

In [ ]:
plot_regression(model, next(iter(train_dataloader)), "Train data")

### Results on test data

In [ ]:
mse, rmse, mae, mre = get_regression_error(model, test_dataloader)
print(f"Test MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, MRE: {mre:.4f}")

In [ ]:
plot_regression(model, next(iter(test_dataloader)), "Test data")